In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN attached:", True)
except Exception as e:
    print("HF_TOKEN missing (benchmarks still run):", e)


In [ ]:
!rm -rf /kaggle/working/arc && git clone --branch stage-a-cpt https://github.com/Nyvo2010/arc.git /kaggle/working/arc
!pip install -q -r /kaggle/working/arc/requirements-kaggle.txt


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="jetmoe/jetmoe-8b", local_dir="/kaggle/working/jetmoe-8b")
print("weights ready")


In [ ]:
import glob, json, os
from pathlib import Path
v = "model_adaptive".split("_")[0]
cands = sorted(glob.glob(f"/kaggle/input/**/{v}/adapter", recursive=True))
cands += sorted(glob.glob("/kaggle/input/arc-tier-b-adapters/{v}/adapter"))
hits = [c for c in cands if (Path(c) / "adapter_config.json").exists()]
assert hits, "No adapter found - is dataset niyuvo/arc-tier-b-adapters attached?"
adapter_dir = hits[0]
print(f"adapter [model_adaptive] ->", adapter_dir)
open("/kaggle/working/adapter.json", "w").write(json.dumps(adapter_dir))


In [ ]:
import json, os, shutil, subprocess, sys, traceback
adapter_dir = json.load(open("/kaggle/working/adapter.json"))
variant = "model_adaptive"
tasks = "arc_easy,piqa"
limits = "arc_easy=10,piqa=10"
ADAPTIVE = [["hard2b", {"bias": 0.38, "k": 12, "halt_threshold": 0.33, "min_gain": 0.07}], ["spreadE", {"bias": 0.52, "k": 8, "halt_threshold": 0.4, "min_gain": 0.03}]]
os.makedirs("/kaggle/working/diag", exist_ok=True)
os.makedirs("/kaggle/output/diag", exist_ok=True)
manifest_path = "/kaggle/output/diag/manifest.json"
manifest = json.load(open(manifest_path)) if os.path.exists(manifest_path) else {}
def save_manifest():
    json.dump(manifest, open(manifest_path, "w"), indent=1)
def run(tag, cmd):
    final = f"/kaggle/output/diag/diag-{tag}.csv"
    if manifest.get(tag) == "ok" and os.path.exists(final):
        print("skip (done):", tag, flush=True)
        return True
    print(">>>", tag, flush=True)
    try:
        subprocess.run(cmd, cwd="/kaggle/working/arc", check=True)
        shutil.copy(f"/kaggle/working/diag/diag-{tag}.csv", final)
        manifest[tag] = "ok"
        save_manifest()
        return True
    except Exception as e:
        manifest[tag] = f"FAIL: {type(e).__name__}: {e}"
        save_manifest()
        print("FAILED:", tag, flush=True)
        traceback.print_exc()
        return False
base = [sys.executable, "scripts/run_benchmarks.py",
        "--base", "/kaggle/working/jetmoe-8b",
        "--model", variant,
        "--adapters", f"{variant}={adapter_dir}",
        "--tasks", tasks, "--limits", limits]
for ml in (1, 2):
    run(f"{variant}-fixed{ml}",
        base + ["--depth", str(ml),
                "--out", f"/kaggle/working/diag/diag-{variant}-fixed{ml}.csv"])
for cname, ckw in ADAPTIVE:
    run(f"{variant}-{cname}",
        base + ["--budgeted", "--max_loops", "4",
                "--controller", json.dumps(ckw),
                "--out", f"/kaggle/working/diag/diag-{variant}-{cname}.csv"])
print("manifest:", json.dumps(manifest, indent=1))


In [ ]:
import csv, glob, json, os
from collections import Counter
def merge_hist(rows):
    c = Counter()
    for r in rows:
        for part in (r.get("halt_hist", "") or "").split(";"):
            if "@" in part:
                k, v = part.split("@")
                try:
                    c[int(k)] += int(v)
                except ValueError:
                    pass
    return ";".join(f"{k}@{c[k]}" for k in sorted(c)) if c else "n/a"
mcq = ["arc_easy", "piqa"]
print(f"{'config':24s} {'acc':>5s} {'loops':>5s} {'GFLOP':>6s} {'nan':>3s}  hist")
for fp in sorted(glob.glob("/kaggle/working/diag/diag-*.csv")):
    rows = [r for r in csv.DictReader(open(fp)) if r["task"] in mcq]
    if not rows:
        print("  (empty: " + os.path.basename(fp) + ")")
        continue
    acc = sum(float(r["acc"]) * 100 for r in rows) / len(rows)
    loops = sum(float(r.get("avg_loops_per_item", 0)) for r in rows) / len(rows)
    gflop = sum(float(r.get("avg_flops_per_item", 0)) for r in rows) / len(rows) / 1e9
    nan = sum(int(float(r.get("nan_halts", 0) or 0)) for r in rows)
    print(f"{os.path.basename(fp)[5:-4]:24s} {acc:5.1f} {loops:5.1f} {gflop:6.1f} {nan:>3}  {merge_hist(rows)}")
